In [ ]:
!pip install -q -U transformers accelerate requests

import os
import json
import glob
import random
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

ln_f = model.model.language_model.norm
lm_head = model.lm_head
print("model loaded")

In [ ]:
_encoder_cache = {}

def _vision_tower_hook(module, inputs, output):
    _encoder_cache["patch_embeddings"] = output.last_hidden_state[:, 1:, :].detach().cpu()

_hook_handle = model.model.vision_tower.register_forward_hook(_vision_tower_hook)
print("hook registered on model.model.vision_tower")

In [ ]:
def find_file(base, filename):
    for root, _, files in os.walk(base):
        if filename in files:
            return os.path.join(root, filename)
    return None

COCO_ROOT = "/kaggle/input/datasets/nadaibrahim/coco2014"
sample_path = find_file(COCO_ROOT, "COCO_val2014_000000000139.jpg")
assert sample_path is not None, f"couldn't find a sample COCO file under {COCO_ROOT} -- check COCO_ROOT"
img_dir = os.path.dirname(sample_path)
print("COCO images live in:", img_dir)

In [ ]:
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_IMAGE_DIRS = ["/kaggle/temp", "/kaggle/temp/VG_100K_2"]

def find_vg_image_path(image_id: str) -> str:
    for d in VG_IMAGE_DIRS:
        candidate = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None

In [ ]:
def resolve_image_path(q: dict) -> str:
    qid = q["question_id"]
    if qid.startswith("pope_"):
        return os.path.join(img_dir, f"{q['image_id']}.jpg")
    elif qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    elif qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)

In [ ]:
HALLUC_TYPE = "attribute"

if HALLUC_TYPE == "relation":
    !wget https://cs.stanford.edu/people/rak248/VG_100K/images.zip -O /kaggle/working/VG_100K.zip
    !wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip -O /kaggle/working/VG_100K_2.zip
    !unzip -q -o /kaggle/working/VG_100K.zip -d /kaggle/temp/
    !unzip -q -o /kaggle/working/VG_100K_2.zip -d /kaggle/temp/
    print("Visual Genome downloaded and extracted")
else:
    print(f"HALLUC_TYPE={HALLUC_TYPE!r} -- Visual Genome not needed, skipping download")

In [ ]:
with open("/kaggle/input/datasets/nocturnalnerd18/vlm-labeled-examples/labeled_examples.json") as f:
    labeled = json.load(f)

random.seed(42)

def select_matched_subset(labeled, h_type, n_per_class=50):
    correct = [l for l in labeled if l["hallucination_type"] == h_type and l["correct"]]
    hallucinated = [l for l in labeled if l["hallucination_type"] == h_type and not l["correct"]]
    return (random.sample(correct, min(n_per_class, len(correct)))
            + random.sample(hallucinated, min(n_per_class, len(hallucinated))))

subset = select_matched_subset(labeled, HALLUC_TYPE)
print(f"{len(subset)} questions selected for {HALLUC_TYPE}, "
      f"{sum(l['correct'] for l in subset)} correct / {sum(not l['correct'] for l in subset)} hallucinated")

In [ ]:
@torch.no_grad()
def cache_discriminative_question_unpooled(image_path, image_id, question_id, question_text):
    image = load_image(image_path)
    inputs = build_inputs(image, text=question_text)
    prompt_len = inputs["input_ids"].shape[-1]

    _encoder_cache.clear()
    gen_ids = model.generate(**inputs, max_new_tokens=5, do_sample=False)
    answer_text = processor.tokenizer.decode(gen_ids[0, prompt_len:], skip_special_tokens=True).strip()

    full_attention_mask = torch.ones_like(gen_ids)
    outputs = model(
        input_ids=gen_ids, pixel_values=inputs["pixel_values"],
        attention_mask=full_attention_mask, output_hidden_states=True,
    )
    image_token_id = model.config.image_token_index
    image_positions = (gen_ids[0] == image_token_id).nonzero(as_tuple=True)[0]

    hidden_states = torch.stack([
        layer[0, image_positions, :].half().cpu() for layer in outputs.hidden_states
    ])  # (33, 576, 4096) -- per-patch, not pooled

    return {
        "image_id": image_id, "question_id": question_id, "question_text": question_text,
        "answer_text": answer_text,
        "hidden_states": hidden_states,
        "encoder_patch_embeddings": _encoder_cache["patch_embeddings"],
    }

In [ ]:
with open("/kaggle/input/datasets/nocturnalnerd18/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)

question_text_by_key = {(q["image_id"], q["question_id"]): q["question_text"] for q in all_questions}
print(len(question_text_by_key), "question texts loaded")

In [ ]:
UNPOOLED_CACHE_DIR = "/kaggle/working/cache/unpooled_diagnostic"
os.makedirs(UNPOOLED_CACHE_DIR, exist_ok=True)

entries = []
skipped = 0
for i, q in enumerate(subset):
    image_path = resolve_image_path(q)
    question_text = question_text_by_key.get((q["image_id"], q["question_id"]))
    if image_path is None or not os.path.exists(image_path) or question_text is None:
        skipped += 1
        continue

    entry = cache_discriminative_question_unpooled(image_path, q["image_id"], q["question_id"], question_text)
    entry["ground_truth_answer"] = q["ground_truth_answer"]
    entry["correct"] = q["correct"]
    entries.append(entry)

    if i == 0:
        torch.save(entry, "/tmp/size_check.pt")
        size_mb = os.path.getsize("/tmp/size_check.pt") / 1e6
        projected_gb = size_mb * len(subset) / 1000
        print(f"first entry: {size_mb:.1f} MB, projected total: ~{projected_gb:.1f} GB")

out_path = os.path.join(UNPOOLED_CACHE_DIR, f"{HALLUC_TYPE}_diagnostic.pt")
torch.save(entries, out_path)
print(f"saved {len(entries)} entries to {out_path}, {skipped} skipped")

In [ ]:
USERNAME = "nocturnalnerd18"
dataset_id = f"unpooled-diagnostic-{HALLUC_TYPE}"

with open(os.path.join(UNPOOLED_CACHE_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": dataset_id,
        "id": f"{USERNAME}/{dataset_id}",
        "licenses": [{"name": "CC0-1.0"}]
    }, f)

!kaggle datasets create -p {UNPOOLED_CACHE_DIR}